# Smoke Test and Live PLAXIS Verification

This notebook is the operator-facing smoke test for the repository.
Run the cells in order.

Flow:
1. Load config and field data.
2. Validate branch detection and search-space setup.
3. Verify PLAXIS connectivity, materials, nodes, and phase names.
4. Write a midpoint parameter set into the model.
5. Run the staged phases, extract O-Cell curves, and compute metrics.
6. Save a diagnostic PNG in `notebooks/_smoke_outputs/`.

Notes:
- The live PLAXIS cells mutate the opened model instance, not the source file on disk.
- This notebook switches the working directory to the repo root so relative paths from the YAML files resolve correctly.
- If a live cell fails, fix the configuration before moving to the next one.


In [ ]:
import os
import socket
import sys
from pathlib import Path

if Path.cwd().name == "notebooks":
    ROOT = Path.cwd().parent
elif (Path.cwd() / "config").exists() and (Path.cwd() / "core").exists():
    ROOT = Path.cwd()
else:
    ROOT = Path.cwd()
    raise RuntimeError(
        "Could not infer repo root. Launch the notebook from the repo root or from notebooks/."
    )

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

try:
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    import yaml
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        "Missing Python dependency for the smoke notebook. Install the project environment from requirements.txt or environment.yml before running this notebook."
    ) from e
from IPython.display import display

from analysis.viz import save_diagnostic_plot
from core.orchestrator import decode_vector, encode_vector, load_yaml
from metrics.interp import load_field_data, segment_branches, resample_with_cycles
from metrics.misfit import compute_all_metrics
from models.hss import HSS
from plaxis.connector import PlaxisConnector
from plaxis.material_writer import _find_material, write_layer_materials
from plaxis.node_resolver import NodeResolver
from plaxis.result_extractor import extract_ocell_results
from plaxis.runner import run_staged_phases

CFG = ROOT / "config"
OUT = ROOT / "notebooks" / "_smoke_outputs"
OUT.mkdir(parents=True, exist_ok=True)

print("Repo root:", ROOT)
print("Smoke output dir:", OUT)


In [ ]:
project = load_yaml(CFG / "project.yaml")
pso_cfg = load_yaml(CFG / "pso.yaml")
profile = load_yaml(CFG / "soil_profile.yaml")
targets = load_yaml(CFG / "targets.yaml")

phase_names = [str(name) for name in project["plaxis"]["phase_names"]]
calc_phase_names = [name for name in phase_names if name.lower() != "initialphase"]
bounds, syms = encode_vector(profile, profile["bounds_mode"])

print("Run name:", project["project"]["run_name"])
print("Base model:", project["plaxis"]["base_model"])
print("Phase names:", phase_names)
print("Calculation phases:", calc_phase_names)
print("Search dimensions:", len(syms))
print("Field CSVs:", targets["field_data"]["csv_files"])


In [ ]:
field = load_field_data(targets)
field_branches = {
    plate: segment_branches(field[plate], targets)
    for plate in ("upper", "lower")
}

for plate in ("upper", "lower"):
    print(f"\n{plate.upper()} field curve")
    display(field[plate].head())
    print("Detected branches:", len(field_branches[plate]))
    for i, br in enumerate(field_branches[plate], start=1):
        print(
            f"  branch {i}: kind={br['kind']}, "
            f"idx=[{br['idx_start']}, {br['idx_end']}], "
            f"confidence={br['confidence']:.2f}"
        )


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), dpi=150)
for ax, plate in zip(axes, ("upper", "lower")):
    df = field[plate]
    ax.plot(df["displacement"], df["load"], "-o", ms=3, lw=1)
    for i, br in enumerate(field_branches[plate], start=1):
        seg = df.iloc[br["idx_start"] : br["idx_end"] + 1]
        ax.plot(
            seg["displacement"],
            seg["load"],
            "o",
            ms=4,
            label=f"br{i} {br['kind']}"
        )
    ax.set_title(f"{plate.title()} field curve")
    ax.set_xlabel("Displacement")
    ax.set_ylabel("Load")
    ax.grid(True, alpha=0.3)
    ax.legend()
plt.tight_layout()


In [ ]:
mid_x = bounds.mean(axis=1)
mid_params = decode_vector(mid_x, syms, profile)

def _port_open(host, port, timeout=1.0):
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except OSError:
        return False

issues = []
for csv_path in targets["field_data"]["csv_files"].values():
    if not Path(csv_path).exists():
        issues.append(f"Missing field CSV: {csv_path}")

exe_path = Path(project["plaxis"]["exe_path"])
base_model_path = Path(project["plaxis"]["base_model"])
input_port = project["plaxis"]["scripting_port_input"]
output_port = project["plaxis"]["scripting_port_output"]
input_ok = _port_open("127.0.0.1", input_port)
output_ok = _port_open("127.0.0.1", output_port)

if not exe_path.exists():
    issues.append(f"PLAXIS executable not found: {project['plaxis']['exe_path']}")
if not base_model_path.exists():
    issues.append(f"Base model not found: {project['plaxis']['base_model']}")
if not input_ok:
    issues.append(f"PLAXIS Input scripting server is not reachable on port {input_port}.")
if not output_ok:
    issues.append(f"PLAXIS Output scripting server is not reachable on port {output_port}.")

print("Midpoint parameter snapshot:")
print(yaml.safe_dump(mid_params, sort_keys=False))
print("PLAXIS executable exists:", exe_path.exists())
print("Base model exists:", base_model_path.exists())
print(f"Input port {input_port}:", input_ok)
print(f"Output port {output_port}:", output_ok)

expected_branches = len(calc_phase_names)
upper_branches = len(field_branches["upper"])
lower_branches = len(field_branches["lower"])
print("Expected branches from PLAXIS config:", expected_branches)
print("Detected upper branches:", upper_branches)
print("Detected lower branches:", lower_branches)
if expected_branches != upper_branches:
    issues.append("Upper branch count does not match configured PLAXIS phases.")
if expected_branches != lower_branches:
    issues.append("Lower branch count does not match configured PLAXIS phases.")

if issues:
    raise RuntimeError("Preflight failed:\n- " + "\n- ".join(issues))


## Live PLAXIS verification

The cells below require:
- PLAXIS 3D 24.x with both Input and Output scripting servers available
- a valid `project.yaml` password and base model path
- phase names, material names, and node definitions that match the actual model


In [ ]:
def port_open(host, port, timeout=1.5):
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except OSError:
        return False

input_port = project["plaxis"]["scripting_port_input"]
output_port = project["plaxis"]["scripting_port_output"]
input_ok = port_open("127.0.0.1", input_port)
output_ok = port_open("127.0.0.1", output_port)

print(f"Input port {input_port}:", input_ok)
print(f"Output port {output_port}:", output_ok)
assert input_ok, f"PLAXIS Input scripting server is not reachable on port {input_port}."
print("Output can remain closed until the phase view step.")


In [ ]:
plx = PlaxisConnector(project["plaxis"]).attach_input()

input_phase_names = [str(ph.Name) for ph in plx.g_i.Phases[:]]

material_checks = {}
for layer in profile["layers"]:
    material_id = layer["plaxis_material_id"]
    material_checks[material_id] = _find_material(plx.g_i, material_id) is not None

print("Input phases:", input_phase_names)
print("Material lookup:", material_checks)
print("Output remains unopened until after calculation + view().")

assert all(material_checks.values()), "One or more PLAXIS material names were not found."
for name in phase_names:
    assert name in input_phase_names, f"Configured phase {name!r} not found in PLAXIS Input."


In [ ]:
write_layer_materials(plx, mid_params, profile, HSS())
print("Successfully wrote midpoint HSS parameters into the open PLAXIS model.")


In [ ]:
smoke_log_path = OUT / "plaxis_smoke.log"
run_log = run_staged_phases(
    plx,
    phase_names=phase_names,
    timeout_min=project["resilience"]["timeout_min_per_run"],
    log_path=smoke_log_path,
)
print(yaml.safe_dump(run_log, sort_keys=False))
print("Log file:", smoke_log_path)
if run_log.get("partial"):
    print("WARNING: staged run ended partial. Continue to extraction to inspect coverage and branch diagnostics.")


In [ ]:
completed_phases = [name for name in run_log.get("phases_run", []) if str(name).lower() != "initialphase"]
assert completed_phases, "No calculation phase completed, so there is nothing to view in Output."
plx.open_output_view(run_log.get("last_phase"))

output_phase_names = [str(ph.Name) for ph in plx.g_o.Phases[:]]
resolver = NodeResolver(plx.g_o)
resolved_nodes = {
    "upper": resolver.resolve(project["ocell"]["upper_node"]),
    "lower": resolver.resolve(project["ocell"]["lower_node"]),
}
print("Output phases:", output_phase_names)
print("Resolved upper node:", resolved_nodes["upper"])
print("Resolved lower node:", resolved_nodes["lower"])

predicted = extract_ocell_results(
    plx,
    phase_names=completed_phases,
    ocell_cfg=project["ocell"],
)

for plate in ("upper", "lower"):
    df = predicted[plate]
    print(f"\n{plate.upper()} predicted curve")
    print("rows:", len(df), "branch ids:", sorted(df["branch_id"].unique()))
    display(df.head())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), dpi=150)
for ax, plate in zip(axes, ("upper", "lower")):
    df = predicted[plate]
    for branch_id in sorted(df["branch_id"].unique()):
        sub = df[df["branch_id"] == branch_id]
        ax.plot(
            sub["displacement"],
            sub["load"],
            marker="o",
            ms=2,
            lw=1,
            label=f"branch {branch_id}"
        )
    ax.set_title(f"{plate.title()} predicted curve")
    ax.set_xlabel("Displacement")
    ax.set_ylabel("Load")
    ax.grid(True, alpha=0.3)
    ax.legend()
plt.tight_layout()


In [ ]:
residuals = {}
per_branch = {}
diag = {}
for plate in ("upper", "lower"):
    residuals[plate], per_branch[plate], diag[plate] = resample_with_cycles(
        predicted[plate],
        field[plate],
        field_branches[plate],
        targets,
    )

metrics = compute_all_metrics(residuals, per_branch, targets, observed=field)
diagnostic_path = OUT / "live_plaxis_diagnostic.png"
save_diagnostic_plot(
    diagnostic_path,
    predicted=predicted,
    field=field,
    field_branches=field_branches,
    per_branch=per_branch,
    diag=diag,
    run_label="plaxis_smoke_test",
    fitness=metrics["primary_total"],
)

print("Per-plate metrics:")
display(pd.DataFrame(metrics["per_plate"]).T)
print("Diagnostics:")
print(yaml.safe_dump(diag, sort_keys=False))
print("Diagnostic PNG:", diagnostic_path)
print("Primary fitness:", metrics["primary_total"])


In [ ]:
if "plx" in globals() and plx is not None:
    plx.close()
    print("PLAXIS connection closed.")
else:
    print("No open PLAXIS connection.")
